# Real Estate Feasibility Engine – Python App
## Step 2: Timeline & curve engine

**What this notebook does**
1. Adds two new engine files next to Step 1's `inputs.py`:
   * `engine/timeline.py` – **WHEN** things happen (calendar, product launch/construction quarters, links, phases, cost windows) = Excel **TIMELINE** sheet
   * `engine/curves.py` – **HOW MUCH per quarter** (Upfront, Even, Front-loaded, Back-loaded, S-curve, Follow …) = Excel **CURVES** sheet
2. Shows the results as tables and charts (calendar, Gantt chart, curve shapes, cost phasing).
3. **Compares every quarter with your Excel FINAL file.**
4. Proves the engine adapts: change one input and everything re-flows.

**Before you start:** Step 1 must have been run once (it created `MyDrive/feasibility_app`). Run cells top to bottom with **Shift + Enter**.

### Cell 1 – Connect to Google Drive and open the project folder

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/feasibility_app'
except ImportError:
    BASE = os.path.abspath('feasibility_app')
os.chdir(BASE)
assert os.path.exists('engine/inputs.py'), 'engine/inputs.py not found – please run the Step 1 notebook first.'
print('Working in:', BASE)
print('Engine files so far:', sorted(f for f in os.listdir('engine') if f.endswith('.py')))

### Cell 2 – Write the timeline engine (`engine/timeline.py`)
* Builds 40 quarter-end dates from the project start, the Indian financial year (Apr–Mar) and project year.
* Works out each active product's **launch, sales end, construction start and end** – applying the scenario delay and sales-speed multiplier, following **links to products listed above**, and falling back to the fixed quarter if a linked product is not active.
* Finds the **last active quarter**, labels each quarter's **phase**, and sets each cost head's **spending window**.

In [ ]:
%%writefile engine/timeline.py
"""
engine/timeline.py  -  STEP 2a: the TIMELINE engine (Python version of the Excel TIMELINE sheet).

It answers one question: WHEN does everything happen?
  * a calendar of 40 quarters (dates, financial year, project year)
  * for every active product: launch, sales end, construction start and construction end
    (after scenario delay, sales-speed multiplier and links between products)
  * the project's last active quarter and a phase label for every quarter
  * for every cost head: the window (start..end quarter) in which it may be spent
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field
from datetime import date, timedelta

from .inputs import Project, HORIZON_Q, MANUAL_CURVES

# ------------------------------------------------------------------ calendar helpers
def quarter_end(start: date, q: int) -> date:
    """End date of quarter q (q = 1 is the quarter ending on the project start date)."""
    months = start.month - 1 + 3 * (q - 1)
    y, m = start.year + months // 12, months % 12 + 1
    nxt = date(y + (m == 12), m % 12 + 1, 1)          # first day of the following month
    return nxt - timedelta(days=1)


def fy_label(d: date) -> str:
    """Indian financial year (April-March), e.g. 30-Jun-2027 -> 'FY 2027-28'."""
    fy_end = d.year + (1 if d.month > 3 else 0)
    return f"FY {fy_end - 1}-{str(fy_end)[-2:]}"


# ------------------------------------------------------------------ result containers
@dataclass
class ProductTiming:
    name: str
    launch: int            # first selling quarter
    sales_end: int         # last selling quarter
    cons_start: int        # first construction quarter
    cons_end: int          # last construction quarter
    sales_n: int           # number of selling quarters
    cons_n: int            # number of construction quarters
    note: str = ""         # plain-English explanation (like Excel Section 5B)


@dataclass
class Timeline:
    quarters: list                                  # [1, 2, ... 40]
    end_dates: list
    start_dates: list
    fy: list
    project_year: list
    products: dict = field(default_factory=dict)    # name -> ProductTiming (active products only)
    last_q: int = 0
    phase: list = field(default_factory=list)
    cost_windows: dict = field(default_factory=dict)  # cost key -> (start, end) or None
    errors: list = field(default_factory=list)


# ------------------------------------------------------------------ product timing (Section 5B logic)
def _event_quarter(t: ProductTiming, duration_input: int, event: str) -> int:
    return {"Launch": t.launch, "Sales end": t.sales_end, "Construction start": t.cons_start,
            "Construction 50%": t.cons_start + math.ceil(duration_input / 2) - 1,
            "Construction end": t.cons_end}[event]


def _offset_text(n: int) -> str:
    return f"{abs(n)} qtr(s) before " if n < 0 else f"{n} qtr(s) after "


def resolve_products(project: Project) -> tuple[dict, list]:
    """Final quarters for every active product, processed top-down so links only point upwards."""
    sc = project.scenario()
    active_names = {p.name for p in project.active_products()}
    resolved, errors = {}, []
    for p in project.products:
        if p.name not in active_names:
            continue
        tm = p.timing
        dur_in = {x.name: x.construction_duration_q for x in project.products}

        def linked(prod, event, offset, fixed_q, label):
            if prod in resolved:                                   # anchor is above AND active
                return _event_quarter(resolved[prod], dur_in[prod], event) + offset, \
                       f"{_offset_text(offset)}{prod} – {event.lower()}"
            return fixed_q + sc.delay_q, f"fixed fallback – {prod} not active"

        # construction start: computed first unless it depends on our own launch
        cons_pre, cons_txt = None, ""
        if tm.cons_mode == "Fixed quarter":
            cons_pre, cons_txt = p.construction_start_q + sc.delay_q, "fixed"
        elif tm.cons_mode == "Linked to another product":
            cons_pre, cons_txt = linked(tm.cons_link_product, tm.cons_link_event, tm.cons_offset, p.construction_start_q, "construction")

        if tm.launch_mode == "Fixed quarter":
            launch, l_txt = p.launch_q + sc.delay_q, "fixed"
        elif tm.launch_mode == "Linked to another product":
            launch, l_txt = linked(tm.launch_link_product, tm.launch_link_event, tm.launch_offset, p.launch_q, "launch")
        else:                                                      # After own construction start
            launch, l_txt = cons_pre + tm.launch_offset, _offset_text(tm.launch_offset) + "own construction start"

        if tm.cons_mode == "After own launch":
            cons_start, cons_txt = launch + tm.cons_offset, _offset_text(tm.cons_offset) + "own launch"
        else:
            cons_start = cons_pre

        sales_n = max(1, math.ceil(p.sales_period_q * sc.sales_period_mult - 1e-9))
        t = ProductTiming(p.name, launch, launch + sales_n - 1, cons_start,
                          cons_start + p.construction_duration_q - 1, sales_n, p.construction_duration_q,
                          f"Launch Q{launch} ({l_txt})  |  Construction Q{cons_start} ({cons_txt})")
        if t.launch < 1 or t.cons_start < 1:
            errors.append(f"[Timing] '{p.name}': launch / construction start falls before Q1.")
        if t.sales_end > HORIZON_Q or t.cons_end > HORIZON_Q:
            errors.append(f"[Timing] '{p.name}': runs beyond Q{HORIZON_Q} (the model horizon).")
        resolved[p.name] = t
    return resolved, errors


# ------------------------------------------------------------------ the full timeline
def build_timeline(project: Project) -> Timeline:
    start = project.info.start_date
    qs = list(range(1, HORIZON_Q + 1))
    ends = [quarter_end(start, q) for q in qs]
    tl = Timeline(quarters=qs, end_dates=ends,
                  start_dates=[quarter_end(start, q - 1) + timedelta(days=1) for q in qs],
                  fy=[fy_label(d) for d in ends], project_year=[f"Y{math.ceil(q / 4)}" for q in qs])

    tl.products, tl.errors = resolve_products(project)
    pts = list(tl.products.values())
    tl.last_q = max([max(t.sales_end, t.cons_end) for t in pts], default=0)
    first_launch = min([t.launch for t in pts], default=0)

    for q in qs:                                                   # phase label per quarter
        selling = any(t.launch <= q <= t.sales_end for t in pts)
        building = any(t.cons_start <= q <= t.cons_end for t in pts)
        if q > tl.last_q:          ph = "Post-project"
        elif selling and building: ph = "Sales + Constr."
        elif selling:              ph = "Sales"
        elif building:             ph = "Construction"
        elif q < first_launch:     ph = "Pre-launch"
        else:                      ph = "Completion / handover"
        tl.phase.append(ph)

    land = project.land
    for c in project.costs:                                        # cost-head spending windows
        curve, s, e = c.curve, c.start_q, c.end_q
        if c.key == "land":                                        # land follows the deal type
            if project.info.deal_type == "JV":
                curve = "Follow Collections"
            else:
                curve, s, e = land.payment_curve, land.payment_start_q, land.payment_end_q
        if not c.applicable:
            tl.cost_windows[c.key] = None
        elif curve in MANUAL_CURVES:
            tl.cost_windows[c.key] = (s, s if curve == "Upfront" else e)
        elif curve == "Follow Project Life":
            tl.cost_windows[c.key] = (1, tl.last_q)
        elif curve == "At Project End":
            tl.cost_windows[c.key] = (tl.last_q, tl.last_q)
        else:                                                      # Follow Construction / Sales / Collections
            tl.cost_windows[c.key] = None
    return tl


def gantt_text(tl: Timeline, quarters: int = 24) -> str:
    """Simple text Gantt: S = selling, C = construction, # = both, . = nothing."""
    lines = ["Quarter".ljust(28) + "".join(f"{q:>3}" for q in range(1, quarters + 1))]
    for t in tl.products.values():
        row = ""
        for q in range(1, quarters + 1):
            s, c = t.launch <= q <= t.sales_end, t.cons_start <= q <= t.cons_end
            row += "  #" if s and c else "  S" if s else "  C" if c else "  ."
        lines.append(t.name[:27].ljust(28) + row)
    return "\n".join(lines)


### Cell 3 – Write the curve engine (`engine/curves.py`)
The five shapes (n = number of quarters, i = position 1 … n):

| Curve | Formula |
|---|---|
| Upfront | 100% in quarter 1 |
| Even | 1 ÷ n |
| Front-loaded | 2(n − i + 1) ÷ [n(n + 1)] |
| Back-loaded | 2i ÷ [n(n + 1)] |
| S-curve | sin(π(i − 0.5)/n) × sin(π/2n) – exact, always totals 100% |

'Follow Construction' uses the products' construction profile (weighted by construction cost; construction heads use only their own category), 'Follow Sales' the sales profile (weighted by revenue), 'Follow Project Life' = Even from Q1 to the last quarter, 'At Project End' = 100% in the last quarter. 'Follow Collections' (JV landowner share) is completed in Step 3.

In [ ]:
%%writefile engine/curves.py
"""
engine/curves.py  -  STEP 2b: the CURVE engine (Python version of the Excel CURVES sheet).

It answers: HOW MUCH of each total falls in each quarter?  Every row is a list of 40 shares
that always adds up to exactly 100% (or 0% when the item is switched off).
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import numpy as np

from .inputs import Project, CRORE, HORIZON_Q, MANUAL_CURVES, CONSTRUCTION_HEADS
from .timeline import Timeline


# ------------------------------------------------------------------ the five curve shapes
def curve_weights(name: str, n: int) -> np.ndarray:
    """Share of the total in each of n quarters (i = 1..n). Always sums to 1."""
    if n <= 0:
        return np.zeros(0)
    i = np.arange(1, n + 1, dtype=float)
    if name == "Upfront":
        w = (i == 1).astype(float)
    elif name == "Even":
        w = np.full(n, 1.0 / n)
    elif name == "Front-loaded":
        w = 2 * (n - i + 1) / (n * (n + 1))
    elif name == "Back-loaded":
        w = 2 * i / (n * (n + 1))
    elif name == "S-curve":                       # exact scaling: sum of sin((2i-1)x) = 1/sin(x), x = pi/2n
        w = np.sin(math.pi * (i - 0.5) / n) * math.sin(math.pi / (2 * n))
    else:
        raise ValueError(f"Unknown curve '{name}'")
    return w


def spread(name: str, start_q: int, end_q: int) -> np.ndarray:
    """Place a curve on the 40-quarter grid from start_q to end_q (inclusive)."""
    out = np.zeros(HORIZON_Q)
    lo, hi = max(start_q, 1), min(end_q, HORIZON_Q)
    if hi < lo:
        return out
    out[lo - 1:hi] = curve_weights(name, hi - lo + 1)          # window clipped to the model (as in Excel)
    return out


# ------------------------------------------------------------------ result container
@dataclass
class Curves:
    revenue: dict = field(default_factory=dict)           # product -> Rs Cr (after scenario price change)
    construction_cost: dict = field(default_factory=dict) # product -> Rs Cr (after scenario cost change)
    sales_pct: dict = field(default_factory=dict)         # product -> 40 shares
    cons_pct: dict = field(default_factory=dict)
    cons_cum: dict = field(default_factory=dict)          # cumulative construction progress (drives CLP)
    sales_profile: np.ndarray = None                      # all products, weighted by revenue
    cons_profile: np.ndarray = None                       # all products, weighted by construction cost
    category_profile: dict = field(default_factory=dict)  # category -> weighted construction profile
    cost_pct: dict = field(default_factory=dict)          # cost key -> 40 shares (None = Follow Collections)
    checks: list = field(default_factory=list)


def _weighted(arrays: dict, weights: dict, names) -> np.ndarray:
    total = sum(weights[n] for n in names)
    if total <= 0:
        return np.zeros(HORIZON_Q)
    return sum(weights[n] * arrays[n] for n in names) / total


def build_curves(project: Project, tl: Timeline) -> Curves:
    sc = project.scenario()
    cv = Curves()
    active = project.active_products()

    for p in active:                                            # per-product sales & construction %
        t = tl.products[p.name]
        cv.revenue[p.name] = p.area_sqft * p.rate * (1 + sc.price_change) / CRORE
        cv.construction_cost[p.name] = p.area_sqft * p.construction_rate * (1 + sc.cost_change) / CRORE
        cv.sales_pct[p.name] = spread(p.sales_curve, t.launch, t.sales_end)
        cv.cons_pct[p.name] = spread(p.construction_curve, t.cons_start, t.cons_end)
        cv.cons_cum[p.name] = np.cumsum(cv.cons_pct[p.name])

    names = [p.name for p in active]
    cv.sales_profile = _weighted(cv.sales_pct, cv.revenue, names)
    cv.cons_profile = _weighted(cv.cons_pct, cv.construction_cost, names)
    for _, _, cat in CONSTRUCTION_HEADS:
        cv.category_profile[cat] = _weighted(cv.cons_pct, cv.construction_cost, [p.name for p in active if p.category == cat])

    cat_of_head = {k: cat for k, _, cat in CONSTRUCTION_HEADS}
    for c in project.costs:                                     # every cost head
        curve = c.curve
        if c.key == "land":
            curve = "Follow Collections" if project.info.deal_type == "JV" else project.land.payment_curve
        if not c.applicable:
            cv.cost_pct[c.key] = np.zeros(HORIZON_Q)
        elif curve == "Follow Construction":
            cv.cost_pct[c.key] = cv.category_profile[cat_of_head[c.key]] if c.key in cat_of_head else cv.cons_profile
        elif curve == "Follow Sales":
            cv.cost_pct[c.key] = cv.sales_profile
        elif curve == "Follow Collections":
            cv.cost_pct[c.key] = None                           # filled by the collections engine (Step 3)
        elif curve == "Follow Project Life":
            cv.cost_pct[c.key] = spread("Even", 1, tl.last_q)
        elif curve == "At Project End":
            cv.cost_pct[c.key] = spread("Upfront", tl.last_q, tl.last_q)
        else:                                                   # Upfront / Even / Front / Back / S-curve
            s, e = tl.cost_windows[c.key]
            cv.cost_pct[c.key] = spread(curve, s, e)

    # checks: every active row must total 100%
    for n in names:
        for label, arr in (("sales", cv.sales_pct[n]), ("construction", cv.cons_pct[n])):
            if abs(arr.sum() - 1) > 1e-9:
                cv.checks.append(f"[Curves] '{n}' {label} % totals {arr.sum():.6f}, not 100%.")
    for k, arr in cv.cost_pct.items():
        c = project.cost(k)
        if arr is None or not c.applicable:
            continue
        if abs(arr.sum() - 1) > 1e-9 and arr.sum() != 0:
            cv.checks.append(f"[Curves] cost head '{c.name}' totals {arr.sum():.6f}, not 100%.")
    return cv


### Cell 4 – Write the Excel comparison helper (`engine/excel_check.py`)
Reads the Excel FINAL file **by row labels**, so it does not depend on cell addresses.

In [ ]:
%%writefile engine/excel_check.py
"""
engine/excel_check.py - reads the Excel FINAL model and compares it with the Python engine.
Works by LABELS (not cell addresses), so it is robust to row positions.
"""
from __future__ import annotations
import numpy as np
import openpyxl

NQ = 40
QCOL0 = 4      # quarter columns start at column D in TIMELINE / CURVES


def _row_values(ws, r):
    return np.array([ws.cell(r, c).value or 0 for c in range(QCOL0, QCOL0 + NQ)], dtype=float)


def read_excel_curves(path: str) -> dict:
    """Returns {'sales': {product: arr}, 'cons': {...}, 'cost': {cost name: arr}, 'profiles': {...}}"""
    wb = openpyxl.load_workbook(path, data_only=True, read_only=False)
    ws = wb["CURVES"]
    out = {"sales": {}, "cons": {}, "cost": {}, "cost_order": [], "profiles": {}}
    block, section = None, None
    for r in range(1, ws.max_row + 1):
        label = ws.cell(r, 2).value
        if not isinstance(label, str):
            continue
        if label.startswith("B.  PRODUCT"):   section = "products"; continue
        if label.startswith("C.  AGGREGATE"): section = "profiles"; continue
        if label.startswith("D.  COST HEAD"): section = "costs"; continue
        if label.startswith("E.  CURVE CHECKS"): break
        if section == "products":
            if label.startswith("   Sales % (curve"):         out["sales"][block] = _row_values(ws, r)
            elif label.startswith("   Construction % (curve"): out["cons"][block] = _row_values(ws, r)
            elif not label.startswith("   "):                   block = label
        elif section == "profiles":
            out["profiles"][label] = _row_values(ws, r)
        elif section == "costs":
            if label.startswith("   Spend %"): out["cost"][block] = _row_values(ws, r); out["cost_order"].append(block)
            elif not label.startswith("   "):   block = label
    return out


def read_excel_timeline(path: str) -> dict:
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["INPUTS"]
    prods = {}
    for r in range(1, ws.max_row + 1):          # product table: active rows have quarters in P..S
        if ws.cell(r, 5).value == "Yes" and isinstance(ws.cell(r, 16).value, (int, float)) and isinstance(ws.cell(r, 2).value, str):
            prods[ws.cell(r, 2).value] = tuple(int(ws.cell(r, c).value) for c in (16, 17, 18, 19))
    return prods


### Cell 5 – Build the timeline for the sample project

In [ ]:
import importlib, pandas as pd
import engine.inputs, engine.timeline, engine.curves, engine.excel_check
for m in (engine.inputs, engine.timeline, engine.curves, engine.excel_check): importlib.reload(m)
from engine.inputs import *
from engine.timeline import build_timeline, gantt_text
from engine.curves import build_curves, curve_weights

project = sample_project()
tl = build_timeline(project)
print('Timing errors:', tl.errors or 'none ✔')
print('Project last active quarter: Q%d  (%s)   →  span %.2f years' % (tl.last_q, tl.end_dates[tl.last_q - 1].strftime('%b %Y'), tl.last_q / 4))

print('\nPRODUCT TIMING (same plain-English logic as Excel Section 5B)')
display(pd.DataFrame([{'Product': t.name, 'Launch': f'Q{t.launch}', 'Sales end': f'Q{t.sales_end}',
                       'Construction': f'Q{t.cons_start}–Q{t.cons_end}', 'How it was decided': t.note} for t in tl.products.values()]))

print('\nCALENDAR (first 24 quarters)')
display(pd.DataFrame({'Quarter': [f'Q{q}' for q in tl.quarters], 'Start': tl.start_dates, 'End': tl.end_dates,
                      'Financial year': tl.fy, 'Project year': tl.project_year, 'Phase': tl.phase}).head(24))
print('\nTEXT GANTT  (S = selling, C = construction, # = both)')
print(gantt_text(tl, 22))

### Cell 6 – Gantt chart

In [ ]:
import plotly.express as px
from engine.timeline import quarter_end
from datetime import timedelta
rows = []
for t in tl.products.values():
    rows.append(dict(Product=t.name, Activity='Selling', Start=quarter_end(project.info.start_date, t.launch - 1) + timedelta(days=1),
                     End=quarter_end(project.info.start_date, t.sales_end)))
    rows.append(dict(Product=t.name, Activity='Construction', Start=quarter_end(project.info.start_date, t.cons_start - 1) + timedelta(days=1),
                     End=quarter_end(project.info.start_date, t.cons_end)))
fig = px.timeline(pd.DataFrame(rows), x_start='Start', x_end='End', y='Product', color='Activity',
                  color_discrete_map={'Selling': '#70AD47', 'Construction': '#5B9BD5'},
                  title=f'{project.info.name} – project timeline')
fig.update_yaxes(autorange='reversed'); fig.update_layout(barmode='group', height=380)
fig.show()

### Cell 7 – Curve gallery (12-quarter window)

In [ ]:
import numpy as np, plotly.graph_objects as go
n = 12
gallery = pd.DataFrame({name: curve_weights(name, n) for name in ['Upfront', 'Even', 'Front-loaded', 'Back-loaded', 'S-curve']},
                       index=[f'i={i}' for i in range(1, n + 1)])
display((gallery.T * 100).assign(Total=lambda d: d.sum(axis=1)).round(2))
fig = go.Figure()
for name in ['Even', 'Front-loaded', 'Back-loaded', 'S-curve']:
    fig.add_trace(go.Scatter(x=list(range(1, n + 1)), y=gallery[name], mode='lines+markers', name=name))
fig.add_trace(go.Scatter(x=list(range(1, n + 1)), y=gallery['S-curve'].cumsum(), name='S-curve cumulative', line=dict(dash='dot'), yaxis='y2'))
fig.update_layout(title='Share of total per quarter – and the cumulative S', yaxis_tickformat='.0%',
                  yaxis2=dict(overlaying='y', side='right', tickformat='.0%'), height=400)
fig.show()

### Cell 8 – Product curves: sales %, construction % and cumulative construction progress

In [ ]:
cv = build_curves(project, tl)
print('Curve checks:', cv.checks or 'every row totals 100% ✔')
Q = [f'Q{q}' for q in range(1, 23)]
table = {}
for name in cv.sales_pct:
    table[(name, 'Sales %')] = cv.sales_pct[name][:22]
    table[(name, 'Construction %')] = cv.cons_pct[name][:22]
    table[(name, 'Construction cumulative')] = cv.cons_cum[name][:22]
display((pd.DataFrame(table, index=Q).T * 100).round(1))

fig = go.Figure()
for name, arr in cv.cons_cum.items():
    fig.add_trace(go.Scatter(x=Q, y=arr[:22], mode='lines', name=name))
fig.update_layout(title='Cumulative construction progress (this drives the CLP milestones in Step 3)', yaxis_tickformat='.0%', height=380)
fig.show()

### Cell 9 – Cost head phasing (share of each cost spent per quarter)

In [ ]:
rows = []
for c in project.costs:
    arr = cv.cost_pct[c.key]
    curve = 'Follow Collections' if (c.key == 'land' and project.info.deal_type == 'JV') else c.curve
    win = tl.cost_windows.get(c.key)
    row = {'Cost head': c.name, 'Applicable': c.applicable, 'Curve': curve, 'Window': f'Q{win[0]}–Q{win[1]}' if win else '—'}
    if arr is None:
        row['Total'] = 'Step 3'
    else:
        row['Total'] = f'{arr.sum():.0%}' if c.applicable else 'Off'
        for q in range(8): row[f'Q{q+1}'] = round(arr[q] * 100, 1)
    rows.append(row)
display(pd.DataFrame(rows).fillna(''))

### Cell 10 – Compare with the Excel FINAL file ✅
Put **Feasibility_Engine_FINAL.xlsx** in Google Drive → `feasibility_app/data/` (or upload it when asked).
Every quarter of every product and cost row is compared. Differences of the order of 0.0000000000000001 are just computer rounding.

In [ ]:
from engine.excel_check import read_excel_curves, read_excel_timeline
xl_path = 'data/Feasibility_Engine_FINAL.xlsx'
if not os.path.exists(xl_path):
    try:
        from google.colab import files
        print('Please choose Feasibility_Engine_FINAL.xlsx from your computer …')
        up = files.upload()
        name = next(iter(up)); os.replace(name, xl_path)
    except Exception as e:
        print('Upload skipped:', e)
if os.path.exists(xl_path):
    xt, xc = read_excel_timeline(xl_path), read_excel_curves(xl_path)
    print('PRODUCT TIMING (launch, sales end, construction start, construction end)')
    for name, t in tl.products.items():
        py = (t.launch, t.sales_end, t.cons_start, t.cons_end)
        print(('✔' if py == xt.get(name) else '✖'), f'{name:28} Python {py}   Excel {xt.get(name)}')
    d_prod = max(max(abs(cv.sales_pct[n] - xc['sales'][n]).max(), abs(cv.cons_pct[n] - xc['cons'][n]).max()) for n in cv.sales_pct)
    d_cost, compared = 0.0, 0
    for i, c in enumerate(project.costs):
        if cv.cost_pct[c.key] is None: continue
        d_cost = max(d_cost, abs(cv.cost_pct[c.key] - xc['cost'][xc['cost_order'][i]]).max()); compared += 1
    print(('✔' if d_prod < 1e-12 else '✖'), f'Product sales & construction curves – largest difference: {d_prod:.1e}')
    print(('✔' if d_cost < 1e-12 else '✖'), f'Cost-head phasing ({compared} rows × 40 quarters) – largest difference: {d_cost:.1e}')
else:
    print('Excel file not available – skipped. (You can add it later and re-run this cell.)')

### Cell 11 – It adapts: change one input, everything re-flows

In [ ]:
import copy
def summary(p, label):
    t = build_timeline(p); c = build_curves(p, t)
    apt = t.products.get('Apartments – Towers A & B')
    ok = '✔' if not (t.errors or c.checks) else '✖'
    print(f'{ok} {label:45} last quarter Q{t.last_q:<3}  products {len(t.products)}  '
          f'apartments {("Q%d–Q%d sell, Q%d–Q%d build" % (apt.launch, apt.sales_end, apt.cons_start, apt.cons_end)) if apt else "—"}')

summary(project, 'Base case')
p = copy.deepcopy(project); p.products[1].construction_duration_q = 24; summary(p, 'Apartment construction 16 → 24 quarters')
p = copy.deepcopy(project); p.info.scenario = 'Worst Case';             summary(p, 'Worst Case scenario')
p = copy.deepcopy(project); p.products[0].launch_q = 5;                 summary(p, 'Villa launch moved Q3 → Q5 (linked products follow)')
p = copy.deepcopy(project); p.info.project_type = 'Apartments';         summary(p, 'Project type = Apartments (villas excluded)')
p = copy.deepcopy(project); p.products[1].clp_plan = 'CLP – 17-Floor Tower'; summary(p, 'Apartments on the 17-floor payment plan')

---
### ✅ Step 2 complete
`engine/` now holds `inputs.py`, `timeline.py`, `curves.py` and `excel_check.py`.

**Next – Step 3:** the collections engine – bookings → payment-plan milestones → cash received, including catch-up payments and the JV landowner share.